In [ ]:
from flask import Flask, request, jsonify
import joblib
import json
import pandas as pd
app = Flask(__name__)
model = joblib.load('personel_ai_model.pkl')
target_le = joblib.load('target_encoder.pkl')
feature_encoders = joblib.load('feature_encoders.pkl')
with open('program_icerikleri.json', 'r', encoding='utf-8') as f:
    bilgi_bankasi = json.load(f)

hedef_map = {
    'Güç ve Kondisyon': 'Güç ve Kondisyon',
    'Kas Kütlesi Kazanımı': 'Kas Kazanımı',
    'Yağ Yakımı': 'Kilo Verme',
    'Rehabilitasyon': 'Sakatlık Sonrası'
}
ekipman_map = {
    'Full Gym': 'Full Gym',
    'Ev Jimnastiği': 'Bodyweight',
    'Minimal Ekipman': 'Dumbbell Only'
}
sakatlik_map = {
    'Yok': 'Yok', 'Diz': 'Diz', 'Omuz': 'Omuz',
    'Bel': 'Bel', 'Kalça': 'Yok'
}

@app.route('/asistan', methods=['POST'])
def asistan_servisi():
    try:
        veri = request.get_json()
        veri['Hedef'] = hedef_map.get(veri['Hedef'], 'Güç ve Kondisyon')
        veri['Ekipman'] = ekipman_map.get(veri['Ekipman'], 'Full Gym')
        veri['Sakatlik_Bolgesi'] = sakatlik_map.get(veri['Sakatlik_Bolgesi'], 'Yok')
        veri['Cinsiyet'] = 0 if veri['Cinsiyet'] == 'Erkek' else 1
        input_df = pd.DataFrame([veri])
        for col, le in feature_encoders.items():
            if col in input_df.columns and col != 'Cinsiyet':
                input_df[col] = le.transform(input_df[col])
        input_df['VKI'] = round(veri['Kilo'] / ((veri['Boy'] / 100) ** 2), 2)
        input_df = input_df[model.feature_names_in_]
        tahmin_id = model.predict(input_df)
        etiket = target_le.inverse_transform(tahmin_id)[0]
        if veri['Cinsiyet'] == 0:
            bmr = 10 * veri['Kilo'] + 6.25 * veri['Boy'] - 5 * veri['Yas'] + 5
        else:
            bmr = 10 * veri['Kilo'] + 6.25 * veri['Boy'] - 5 * veri['Yas'] - 161
        kalori = int(bmr * veri['Aktivite_Seviyesi'])
        detay = bilgi_bankasi.get(etiket, {})
        return jsonify({"status": "success", "program_etiketi": etiket, "kalori": kalori, "detaylar": detay})
    except Exception as e:
        return jsonify({"status": "error", "message": str(e)})

if __name__ == '__main__':
    app.run(host='0.0.0.0', port=5000, debug=False, use_reloader=False)

 * Serving Flask app '__main__'
 * Debug mode: off


 * Running on all addresses (0.0.0.0)
 * Running on http://127.0.0.1:5000
 * Running on http://10.245.0.189:5000
Press CTRL+C to quit
127.0.0.1 - - [15/Jun/2026 12:31:21] "POST /asistan HTTP/1.1" 200 -
